In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

**Loading the cleaned data**

In [2]:
df=pd.read_csv('cleaned_data')

In [3]:
df.head()

,Price,city,Owner,Fuel_Type,Transmission,Insurance_Type,Brand,Model,Rate,Kms,years_old,RTO_state_code
0,32.837858,3686,1,3,0,0,2.479394,34.316610,4.0,34854,14,28.455439
1,36.756224,3686,2,3,0,0,2.479394,35.324112,4.2,39541,11,28.455439
2,37.111751,3686,2,3,0,0,2.479394,35.324112,4.3,23233,10,28.455439
3,34.658879,3686,1,3,0,0,2.480042,35.451408,4.4,27748,11,28.455439
4,36.095916,3686,2,3,0,0,2.480042,35.451408,4.4,12238,7,31.784596


In [4]:
x=df.drop('Price',axis=1)
y=df['Price']

In [5]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=45)

In [6]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train=sc.fit_transform(x_train)

In [7]:
x_test=sc.transform(x_test)

# 1. Linear Regression Model

In [8]:
from sklearn.linear_model import LinearRegression
model_lr=LinearRegression()
model_lr.fit(x_train,y_train)
print("intercept:",model_lr.intercept_)
print("coefficents:",model_lr.coef_)

intercept: 37.229006840798334
coefficents: [ 0.48250407 -0.17984085 -0.53101269  0.37099928  0.          0.64481656
  2.48855839  0.59321481 -0.30345704 -1.82533054  0.16585297]


In [9]:
ypred_train=model_lr.predict(x_train)
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
r2_train=r2_score(y_train,ypred_train)
print("r2_train:",r2_train)
cv_train=cross_val_score(model_lr,x_train,y_train,cv=5).mean()
print("cv_train:",cv_train)
ypred_test=model_lr.predict(x_test)
#evaluation on test data
r2_test=r2_score(y_test,ypred_test)
print('r2_test',r2_test)

r2_train: 0.8776537220019783
cv_train: 0.8774161997910518
r2_test 0.8639189830861997


# 2. Lasso Regression

In [10]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso
estimator=Lasso()
param_grid={'alpha':list(range(1,100))}
model_hp=GridSearchCV(estimator,param_grid,cv=5,scoring='r2')
model_hp.fit(x_train,y_train)
model_hp.best_params_

{'alpha': 1}

**final Lasso Model**

In [11]:
lasso_best=Lasso(alpha=1)
lasso_best.fit(x_train,y_train)
ypred_train=lasso_best.predict(x_train)
print('Train R2:',r2_score(y_train,ypred_train))
print('cv score',cross_val_score(lasso_best,x_train,y_train,cv=5).mean())
ypred_test=lasso_best.predict(x_test)
print('Test R2:',r2_score(y_test,ypred_test))

Train R2: 0.7915160377060294
cv score 0.7913884816918212
Test R2: 0.7756358088124566


# Ridge Regression Model

In [13]:
from sklearn.linear_model import Ridge
estimator=Ridge()
param_grid={"alpha":(range(1,10))}
model_hp=GridSearchCV(estimator,param_grid,cv=5,scoring='r2')
model_hp.fit(x_train,y_train)
model_hp.best_params_

{'alpha': 7}

In [14]:
ridge_best=Ridge(alpha=7)
ridge_best.fit(x_train,y_train)
ypred_train=ridge_best.predict(x_train)
print('Train R2:',r2_score(y_train,ypred_train))
print('cv_score:',cross_val_score(ridge_best,x_train,y_train,cv=5).mean())
ypred_test=ridge_best.predict(x_test)
print('Test R2:',r2_score(y_test,ypred_test))

Train R2: 0.8776536952762184
cv_score: 0.8774162361509552
Test R2: 0.8639192121720438


# 4. ElasticNet Regression Model

In [15]:
from sklearn.linear_model import ElasticNet
enr=ElasticNet()
enr.fit(x_train,y_train)
ypred_train=enr.predict(x_train)
ypred_test=enr.predict(x_test)
print('Train R2:',enr.score(x_train,y_train))
print('Test R2:',enr.score(x_test,y_test))
print('cv score:',cross_val_score(enr,x_train,y_train,cv=5).mean())


Train R2: 0.7908228777345134
Test R2: 0.7785305415461695
cv score: 0.7906160620798433


In [16]:
estimator=ElasticNet()
param_grid={'alpha':[0.1,0.2,1.2,3,5,10],'l1_ratio':[0.1,0.5,0.75,0.95,0.1]}
model_hp=GridSearchCV(estimator,param_grid,cv=5)
model_hp.fit(x_train,y_train)
model_hp.best_params_

{'alpha': 0.1, 'l1_ratio': 0.75}

In [17]:
enr_best=ElasticNet(alpha=0.1,l1_ratio=0.75)
enr_best.fit(x_train,y_train)
ypred_train=enr_best.predict(x_train)
ypred_test=enr_best.predict(x_test)
print('Train R2:',enr_best.score(x_train,y_train))
print('Test R2:',enr_best.score(x_test,y_test))
print('cv score:',cross_val_score(enr_best,x_train,y_train,cv=5).mean())


Train R2: 0.8762394093163113
Test R2: 0.862534768231544
cv score: 0.8760119080819015


# 5. Polynomial Regression Model

In [18]:
from sklearn.preprocessing import PolynomialFeatures
polynomial_converter=PolynomialFeatures(degree=3)
x_train_poly=pd.DataFrame(polynomial_converter.fit_transform(x_train))

In [19]:
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(x_train_poly,y_train)


LinearRegression()

In [20]:
# prediction on train data
ypred_train=model.predict(x_train_poly)
# train r2_score
from sklearn.metrics import r2_score
print('train_r2:',r2_score(y_train,ypred_train))
#cv score
from sklearn.model_selection import cross_val_score
print('cv_score:',cross_val_score(model,x_train_poly,y_train,cv=5).mean())
#increasing degree on test data
x_test_poly=pd.DataFrame(polynomial_converter.fit_transform(x_test))
ypred_test=model.predict(x_test_poly)
print('test_r2:',r2_score(y_test,ypred_test))

train_r2: 0.9094577590119752
cv_score: 0.8990425099123411
test_r2: 0.8630795075468576


# 6. KNN Regression Model

In [21]:
from sklearn.neighbors import KNeighborsRegressor
knn=KNeighborsRegressor()
param_grid={'n_neighbors':list(range(1,20))}
knn_search=GridSearchCV(knn,param_grid,cv=5)
knn_search.fit(x_train,y_train)
knn_search.best_params_

{'n_neighbors': 9}

In [22]:
knn_best=KNeighborsRegressor(n_neighbors=9)
knn_best.fit(x_train,y_train) 
ypred_train=knn_best.predict(x_train)
print('Train R2:',r2_score(y_train,ypred_train))
print('cv score:',cross_val_score(knn_best,x_train,y_train,cv=5).mean())
ypred_test=knn_best.predict(x_test)
print('Test R2:',r2_score(y_test,ypred_test))

Train R2: 0.919685922147098
cv score: 0.894234297524361
Test R2: 0.8896908228337876


# 7. SVM Regression Model

In [61]:
from sklearn.svm import SVR
svm=SVR()
svm.fit(x_train,y_train)
ypred_train=svm.predict(x_train)
ypred_test=svm.predict(x_test)
print('Train R2:',r2_score(y_train,ypred_train))
print('cv score:',cross_val_score(svm,x_train,y_train,cv=5).mean())
print('Test R2:',r2_score(y_test,ypred_test))

Train R2: 0.9094188662487572


KeyboardInterrupt: 

In [ ]:
param_grid={'C':[0.01,0.1,1,10,100],'kernel':['linear','rbf','sigmoid','poly']}
grid=GridSearchCV(svm,param_grid,cv=5)
grid.fit(x_train,y_train)
grid.best_params_

In [ ]:
svm_best=SVC(kernel='',C=)
svm_best.fit(x_train,y_train)
ypred_train=svm_best.predict(x_train)
ypred_test=svm_best.predict(x_test)
print('Train R2:',r2_score(y_train,ypred_train))
print('cv score:',cross_val_score(svm,x_train,y_train,cv=5).mean())
print('Test R2:',r2_score(y_test,ypred_test))

# 8. Decision Tree Regression Model 

In [23]:
from sklearn.tree import DecisionTreeRegressor
dt=DecisionTreeRegressor()
dt.fit(x_train,y_train)
ypred_train=dt.predict(x_train)
ypred_test=dt.predict(x_test)
print('Train R2:',r2_score(y_train,ypred_train))
print('cv score:',cross_val_score(dt,x_train,y_train,cv=5).mean())
print('Test R2:',r2_score(y_test,ypred_test))

Train R2: 0.9999994423846267
cv score: 0.8849376963685079
Test R2: 0.8764310436933433


In [24]:
param_grid={'max_depth':list(range(1,21))}
grid=GridSearchCV(dt,param_grid,cv=5)
grid.fit(x_train,y_train)
grid.best_params_

{'max_depth': 10}

In [27]:
dt_best=DecisionTreeRegressor(max_depth=10,random_state=0)
dt_best.fit(x_train,y_train)
ypred_train=dt_best.predict(x_train)
print(r2_score(y_train,ypred_train))
print(cross_val_score(dt_best,x_train,y_train,cv=5).mean())
ypred_test=dt_best.predict(x_test)
print(r2_score(y_test,ypred_test))

0.9384342641715218
0.9062571806312617
0.8984212141065215


# 9. Random Forest Regression Model

In [28]:
from sklearn.ensemble import RandomForestRegressor
rf=RandomForestRegressor()
rf.fit(x_train,y_train)
ypred_train=rf.predict(x_train)
print(r2_score(y_train,ypred_train))
print(cross_val_score(rf,x_train,y_train,cv=5).mean())
ypred_test=rf.predict(x_test)
print(r2_score(y_test,ypred_test))

0.9919287534123282
0.9404122399194424
0.9297792439057326


In [29]:
param_grid_rf={'n_estimators':list(range(15,21))}
p_rf=GridSearchCV(rf,param_grid_rf,cv=5)
p_rf.fit(x_train,y_train)
p_rf.best_params_

{'n_estimators': 20}

In [30]:
rf_best=RandomForestRegressor(n_estimators=20,random_state=0)
rf_best.fit(x_train,y_train)
ypred_train=rf_best.predict(x_train)
print(r2_score(y_train,ypred_train))
print(cross_val_score(rf_best,x_train,y_train,cv=5).mean())
ypred_test=rf_best.predict(x_test)
print(r2_score(y_test,ypred_test))

0.9903920589196283
0.9378173516215347
0.9276918099948738


# 10. Gradient Boost Regression

In [31]:
from sklearn.ensemble import GradientBoostingRegressor
gb=GradientBoostingRegressor()
gb.fit(x_train,y_train)
ypred_train=gb.predict(x_train)
print('train_accuracy:',r2_score(y_train,ypred_train))
print('cv:',cross_val_score(gb,x_train,y_train,cv=5).mean())
ypred_test=gb.predict(x_test)
print('test_accuracy:',r2_score(y_test,ypred_test))

train_accuracy: 0.9250072647377224
cv: 0.9212747577635788
test_accuracy: 0.9105723640543653


In [39]:
param_grid_gb={'learning_rate':[0.4,0.7,0.9],'n_estimators':list(range(34,40))}
p_gb=GridSearchCV(gb,param_grid_gb,cv=5)
p_gb.fit(x_train,y_train)
p_gb.best_params_

{'learning_rate': 0.7, 'n_estimators': 39}

In [42]:
gb_best=GradientBoostingRegressor(learning_rate= 0.7,n_estimators=39)
gb_best.fit(x_train,y_train)
ypred_train=gb_best.predict(x_train)
print('train_accuracy:',r2_score(y_train,ypred_train))
print('cv:',cross_val_score(gb_best,x_train,y_train,cv=5).mean())

ypred_test=gb_best.predict(x_test)
print('test_accuracy:',r2_score(y_test,ypred_test))

train_accuracy: 0.9302888367547102
cv: 0.9250348724667848
test_accuracy: 0.9141835311601922


# 11. Xgboost Regression Model

In [36]:
from xgboost import XGBRegressor
xg=XGBRegressor(random_state=0)
xg.fit(x_train,y_train)
ypred_train=xg.predict(x_train)
print('train_accuracy:',r2_score(y_train,ypred_train))
print('cv:',cross_val_score(xg,x_train,y_train,cv=5).mean())

ypred_test=xg.predict(x_test)
print('test_accuracy:',r2_score(y_test,ypred_test))

train_accuracy: 0.96813295761171
cv: 0.9460926959138497
test_accuracy: 0.9354998032648885


 No overfitting so no need for gridsearchcv

In [62]:
param_grid_xg={'gamma':[0.01,0.15,0.3,0.5,1],'max_depth':[4,5,6],'n_estimators':list(range(24,35))}
p_xg=GridSearchCV(xg,param_grid_xg,cv=5)
p_xg.fit(x_train,y_train)
p_xg.best_params_

{'gamma': 1, 'max_depth': 6, 'n_estimators': 34}

In [63]:
xg_best=XGBRegressor(gamma=1,max_depth=6,n_estimators=34)
xg_best.fit(x_train,y_train)
ypred_train=xg_best.predict(x_train)
ypred_test=xg_best.predict(x_test)
print("train_accuracy:",r2_score(y_train,ypred_train))
print('cv:',cross_val_score(xg_best,x_train,y_train,cv=5).mean())
print('test_accuracy:',r2_score(y_test,ypred_test))

train_accuracy: 0.9540905069817889
cv: 0.9406251398831532
test_accuracy: 0.929525169272206


# 12. Artificial Neural Networks

In [44]:
from keras.models import Sequential
#model
ann=Sequential()

In [45]:
from keras.layers import Dense
#adding input layer 
ann.add(Dense(input_dim=11,units=22,kernel_initializer='uniform',activation='relu'))

C:\Users\npdre\anaconda3\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [46]:
ann.add(Dense(units=1,kernel_initializer='uniform',activation='relu'))

In [54]:
ann.compile(optimizer='adam',loss='binary_crossentropy',metrics=['r2_score'])

In [55]:
ann.fit(x_train,y_train,epochs=50,batch_size=32)

Epoch 1/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: -577.3273 - r2_score: -43.7445
Epoch 2/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: -576.9612 - r2_score: -44.5989
Epoch 3/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: -578.2695 - r2_score: -43.9993
Epoch 4/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: -577.4274 - r2_score: -44.4616
Epoch 5/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: -578.1633 - r2_score: -44.7554
Epoch 6/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: -578.3452 - r2_score: -44.7053
Epoch 7/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: -577.8401 - r2_score: -44.9573
Epoch 8/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: -578.4031 - r2_score: -44.6011
Epoch 9/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: -577.9515 - r2_score: -44.1545
Epoch 10/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: -577.8505 - r2_score: -44.0640
Epoch 11/50
804/804 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: -577.0504 - r2_score: -44.15

In [56]:
ypred_test=ann.predict(x_test)
ypred_test=(ypred_test.round())

201/201 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [60]:
from sklearn.metrics import r2_score
print('test_accuracy:',r2_score(y_test,ypred_test))

test_accuracy: -43.772390751305224


**ANN model is Not working for this dataset**

# Final best model is XgBoostRegressor with :
**train_accuracy: 0.96813295761171**

**cv: 0.9460926959138497**

**test_accuracy: 0.9354998032648885**